# 📐 Modul 02 Hands-on: Feature Extraction & Classical Computer Vision

Sebelum era Deep Learning, computer vision mengandalkan ekstraksi fitur manual berbasis matematika spasial.
Pada hands-on ini, Anda akan mempraktikkan:
1. **The Working Blackbox**: Pipeline deteksi tepi Sobel & Canny.
2. **Anatomical Dissection**: Konvolusi 2D manual dengan kernel Sobel X & Sobel Y.
3. **The Sabotage Lab**: Menjalankan deteksi tepi TANPA Gaussian blur (derau parah) vs DENGAN Gaussian blur (tepi bersih).
4. **Operasi Morfologi**: Erosi & Dilasi untuk membersihkan mask biner.
5. **Tantangan**: Membangun shape classifier sederhana berbasis perimeter & area ratio.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent / "shared"))
from cv_utils import show_tensor

print("✅ Setup lingkungan selesai!")

## 1. Menyiapkan Gambar Uji Sintetis dengan Noise

In [ ]:
# Buat gambar sintetis bentuk geometris dengan noise bintik
np.random.seed(42)
img_clean = np.zeros((200, 200), dtype=np.float32)
img_clean[40:160, 40:160] = 0.8  # Persegi
rr, cc = np.ogrid[:200, :200]
circle_mask = ((rr - 100)**2 + (cc - 100)**2) < 30**2
img_clean[circle_mask] = 0.2  # Lingkaran di dalam

# Tambahkan Gaussian noise
noise = np.random.normal(0, 0.15, (200, 200)).astype(np.float32)
img_noisy = np.clip(img_clean + noise, 0.0, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_clean, cmap="gray"); axes[0].set_title("Gambar Bersih"); axes[0].axis("off")
axes[1].imshow(img_noisy, cmap="gray"); axes[1].set_title("Gambar Ber-Noise (Tantangan Nyata)"); axes[1].axis("off")
plt.tight_layout(); plt.show()

## 2. Anatomical Dissection: Konvolusi Spasial Manual dengan Kernel Sobel
Operator Sobel menghitung turunan parsial horizontal ($G_x$) dan vertikal ($G_y$).

In [ ]:
tensor_noisy = torch.tensor(img_noisy).unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]

# Kernel Sobel X dan Sobel Y
sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)

grad_x = F.conv2d(tensor_noisy, sobel_x, padding=1)
grad_y = F.conv2d(tensor_noisy, sobel_y, padding=1)
grad_magnitude = torch.sqrt(grad_x**2 + grad_y**2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(grad_x.squeeze().numpy(), cmap="gray"); axes[0].set_title("Gradien X (Tepi Vertikal)"); axes[0].axis("off")
axes[1].imshow(grad_y.squeeze().numpy(), cmap="gray"); axes[1].set_title("Gradien Y (Tepi Horizontal)"); axes[1].axis("off")
axes[2].imshow(grad_magnitude.squeeze().numpy(), cmap="hot"); axes[2].set_title("Magnitudo Gradien Gabungan"); axes[2].axis("off")
plt.tight_layout(); plt.show()

## 3. The Sabotage Lab: Deteksi Tepi Tanpa vs Dengan Gaussian Blur
Apa yang terjadi jika kita tidak meredam frekuensi tinggi (noise) sebelum ekstraksi tepi?

In [ ]:
# Kernel Gaussian Blur 5x5
gaussian_kernel = torch.tensor([
    [1, 4, 7, 4, 1],
    [4, 16, 26, 16, 4],
    [7, 26, 41, 26, 7],
    [4, 16, 26, 16, 4],
    [1, 4, 7, 4, 1]
], dtype=torch.float32).view(1, 1, 5, 5)
gaussian_kernel = gaussian_kernel / gaussian_kernel.sum()

# Kasus 1: SABOTASE (Tanpa Blur langsung thresholding tepi)
edge_raw = (grad_magnitude > 0.6).float()

# Kasus 2: DENGAN Gaussian Blur terlebih dahulu
img_blurred = F.conv2d(tensor_noisy, gaussian_kernel, padding=2)
grad_x_b = F.conv2d(img_blurred, sobel_x, padding=1)
grad_y_b = F.conv2d(img_blurred, sobel_y, padding=1)
grad_mag_b = torch.sqrt(grad_x_b**2 + grad_y_b**2)
edge_clean = (grad_mag_b > 0.4).float()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(edge_raw.squeeze().numpy(), cmap="gray")
axes[0].set_title("❌ SABOTASE: Tanpa Blur\n(Kekacauan False Edges!)", color="red", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(edge_clean.squeeze().numpy(), cmap="gray")
axes[1].set_title("✅ BENAR: Gaussian Blur + Sobel\n(Tepi Objek Jelas & Bersih)", color="green", fontweight="bold")
axes[1].axis("off")
plt.tight_layout(); plt.show()

## 4. Morfologi Citra: Erosi & Dilasi
Erosi menyusutkan objek terang, Dilasi memperbesar objek terang.

In [ ]:
# Implementasi dilasi menggunakan max-pooling 3x3
dilated = F.max_pool2d(edge_clean, kernel_size=3, stride=1, padding=1)
# Implementasi erosi menggunakan negative max-pooling
eroded = -F.max_pool2d(-edge_clean, kernel_size=3, stride=1, padding=1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(edge_clean.squeeze().numpy(), cmap="gray"); axes[0].set_title("Original Binary Edge"); axes[0].axis("off")
axes[1].imshow(dilated.squeeze().numpy(), cmap="gray"); axes[1].set_title("Dilasi (Menebalkan Tepi)"); axes[1].axis("off")
axes[2].imshow(eroded.squeeze().numpy(), cmap="gray"); axes[2].set_title("Erosi (Menipiskan Tepi)"); axes[2].axis("off")
plt.tight_layout(); plt.show()

## 5. 🏆 Tantangan Mandiri: Menghitung Rasio Luas & Keliling
Gunakan erosi/dilasi untuk menghitung perimeter dan area objek, lalu simpulkan apakah objek tersebut lingkaran atau kotak!